# Ingestion Control Setup

This notebook creates the persistent Delta control tables used to monitor and audit source ingestion for the Tuklas Pinas data pipeline.

The `01_ingestion` schema stores pipeline-run metadata, source-file and source-table processing results, and ingestion errors.

Actual source data is written separately to the Bronze layer under `tuklas_dev`.`02_bronze`.

The control tables created here are designed to support:

* `01_ingest_psa`
* `02_ingest_psgc`
* `03_ingest_dot`

### Step 1 — Configuration

Define the Unity Catalog identifiers and a helper for fully qualified ingestion-table names.

The `01_ingestion` schema already exists and must **not** be dropped or recreated.

In [0]:
CATALOG = "tuklas_dev"
INGESTION_SCHEMA = "01_ingestion"
BRONZE_SCHEMA = "02_bronze"

def ingestion_table(table_name):
    """Return the fully qualified, backtick-quoted identifier for an ingestion control table."""
    return f"`{CATALOG}`.`{INGESTION_SCHEMA}`.`{table_name}`"

# Verify that the ingestion schema already exists (do NOT create or drop it)
schema_exists = spark.sql(
    f"SELECT 1 FROM `{CATALOG}`.information_schema.schemata "
    f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{INGESTION_SCHEMA}'"
).count()

assert schema_exists == 1, (
    f"Schema `{CATALOG}`.`{INGESTION_SCHEMA}` does not exist. "
    "Create it manually before running this notebook."
)
print(f"✓ Schema `{CATALOG}`.`{INGESTION_SCHEMA}` verified.")
print(f"  Bronze schema: `{CATALOG}`.`{BRONZE_SCHEMA}` (for reference only — not modified here)")

✓ Schema `tuklas_dev`.`01_ingestion` verified.
  Bronze schema: `tuklas_dev`.`02_bronze` (for reference only — not modified here)


### Table 1 — `ingestion_run_log`

**Purpose:** One row represents one execution of an ingestion notebook.

This table summarizes the **entire** notebook run — not individual datasets. A single PSA execution may process DS\_01 (10 Bronze tables), DS\_02 (3 tables), PSY/DS\_08–17, DS\_18 Excel, DS\_18 PDF, and multiple CSV files; all of that is captured in one row here.

| Column | Type | Description |
| --- | --- | --- |
| `run_id` | STRING | Unique run identifier |
| `notebook_name` | STRING | e.g. `01_ingest_psa` |
| `source_system` | STRING | e.g. `PSA` |
| `source_folder` | STRING | Google Drive folder URL |
| `started_at` | TIMESTAMP | Run start time |
| `completed_at` | TIMESTAMP | Run end time |
| `files_discovered` | BIGINT | Files found in source |
| `files_ingested` | BIGINT | Files successfully ingested |
| `files_skipped` | BIGINT | Files intentionally skipped |
| `files_failed` | BIGINT | Files that failed |
| `tables_created` | BIGINT | Bronze tables created/refreshed |
| `status` | STRING | `STARTED`, `SUCCESS`, `PARTIAL_SUCCESS`, `FAILED` |
| `notes` | STRING | Free-text notes |

No restrictive `CHECK` constraints are added to avoid compatibility issues in Databricks Free Edition.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ingestion_table('ingestion_run_log')} (
    run_id            STRING,
    notebook_name     STRING,
    source_system     STRING,
    source_folder     STRING,
    started_at        TIMESTAMP,
    completed_at      TIMESTAMP,
    files_discovered  BIGINT,
    files_ingested    BIGINT,
    files_skipped     BIGINT,
    files_failed      BIGINT,
    tables_created    BIGINT,
    status            STRING,
    notes             STRING
) USING DELTA
""")

# ── Validate ──
run_log_df = spark.table(ingestion_table("ingestion_run_log"))
print(f"✓ Table {ingestion_table('ingestion_run_log')} created / verified.")
print(f"Rows: {run_log_df.count()}")
run_log_df.printSchema()
display(run_log_df.limit(10))

✓ Table `tuklas_dev`.`01_ingestion`.`ingestion_run_log` created / verified.
Rows: 0
root
 |-- run_id: string (nullable = true)
 |-- notebook_name: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_folder: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- files_discovered: long (nullable = true)
 |-- files_ingested: long (nullable = true)
 |-- files_skipped: long (nullable = true)
 |-- files_failed: long (nullable = true)
 |-- tables_created: long (nullable = true)
 |-- status: string (nullable = true)
 |-- notes: string (nullable = true)



run_id,notebook_name,source_system,source_folder,started_at,completed_at,files_discovered,files_ingested,files_skipped,files_failed,tables_created,status,notes


### Table 2 — `file_manifest`

This is the most important control table. It stores the result of processing **each logical source/output table** during an ingestion run.

One manifest row = one logical ingestion output. One physical file may produce **multiple** Bronze tables, so `file_manifest` can have many rows for the same `source_file`, differentiated by `source_sheet` and `destination_table`.

**Examples:**

* `DS_01_PSA_PTSA_Statistical_Tables_2000_2025.xlsx` → 10 rows (one per sheet, `psa_ds_01_table_01` through `psa_ds_01_table_10`)
* `DS_02_PSA_Tourism_Statistical_Classification_2025.xlsx` → 3 rows (`psa_ds_02_metadata`, `psa_ds_02_tourism_industries`, `psa_ds_02_tourism_products`)
* `DS_08_to_DS_17_PSA_PSY_Tables_8_1_to_8_10.xlsx` → 10 rows (T8\_1→DS\_08→`psa_ds_08` through T8\_10→DS\_17→`psa_ds_17`)
* DS\_18 Excel and PDF share `source_dataset_id = DS_18` but produce different Bronze tables (`psa_ds_18` vs `psa_ds_18_pdf`)

The combination of `run_id` + `source_dataset_id` + `source_file` + `source_sheet` + `destination_table` distinguishes processing records. No primary-key constraint is enforced.

**Status values:** `DISCOVERED`, `INGESTED`, `SKIPPED`, `NEEDS_REVIEW`, `FAILED` — compatible with the existing PSA notebook.

| Column | Type | Description |
| --- | --- | --- |
| `run_id` | STRING | Foreign reference to `ingestion_run_log.run_id` |
| `notebook_name` | STRING | Notebook that produced this record |
| `source_dataset_id` | STRING | e.g. `DS_01`, `DS_08` |
| `source_system` | STRING | e.g. `PSA` |
| `source_file` | STRING | Physical file name from Google Drive |
| `source_sheet` | STRING | Worksheet name (NULL for CSV/PDF) |
| `source_path` | STRING | Full source URL/path |
| `destination_table` | STRING | Bronze table name |
| `row_count` | BIGINT | Rows written to Bronze |
| `column_count` | BIGINT | Columns in Bronze table |
| `status` | STRING | Processing status |
| `discovered_at` | TIMESTAMP | When the file was discovered |
| `ingested_at` | TIMESTAMP | When the file was ingested |
| `notes` | STRING | Free-text notes |

**Compatibility with current PSA notebook:**

| PSA summary / Bronze metadata field | `file_manifest` column |
| --- | --- |
| `source_dataset_id` / `_source_dataset_id` | `source_dataset_id` |
| `source_file` / `_source_file` | `source_file` |
| `source_sheet` / `_source_sheet` | `source_sheet` |
| `bronze_table` | `destination_table` |
| `row_count` | `row_count` |
| `column_count` | `column_count` |
| `status` | `status` |
| `notes` | `notes` |
| `_source_path` | `source_path` |
| `_source_system` | `source_system` |
| `_ingested_at` | `ingested_at` |

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ingestion_table('file_manifest')} (
    run_id             STRING,
    notebook_name      STRING,
    source_dataset_id  STRING,
    source_system      STRING,
    source_file        STRING,
    source_sheet       STRING,
    source_path        STRING,
    destination_table  STRING,
    row_count          BIGINT,
    column_count       BIGINT,
    status             STRING,
    discovered_at      TIMESTAMP,
    ingested_at        TIMESTAMP,
    notes              STRING
) USING DELTA
""")

# ── Validate ──
manifest_df = spark.table(ingestion_table("file_manifest"))
print(f"✓ Table {ingestion_table('file_manifest')} created / verified.")
print(f"Rows: {manifest_df.count()}")
manifest_df.printSchema()
display(manifest_df.limit(10))

✓ Table `tuklas_dev`.`01_ingestion`.`file_manifest` created / verified.
Rows: 0
root
 |-- run_id: string (nullable = true)
 |-- notebook_name: string (nullable = true)
 |-- source_dataset_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- source_path: string (nullable = true)
 |-- destination_table: string (nullable = true)
 |-- row_count: long (nullable = true)
 |-- column_count: long (nullable = true)
 |-- status: string (nullable = true)
 |-- discovered_at: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- notes: string (nullable = true)



run_id,notebook_name,source_dataset_id,source_system,source_file,source_sheet,source_path,destination_table,row_count,column_count,status,discovered_at,ingested_at,notes


### Table 3 — `ingestion_errors`

**Purpose:** Store actual ingestion exceptions and technical failures separately from the normal file manifest.

This table supports errors such as Google Drive read failures, CSV/Excel parsing failures, worksheet parsing failures, Delta write failures, unexpected schema issues, and binary PDF ingestion failures.

No errors are intentionally generated to populate this table.

| Column | Type | Description |
| --- | --- | --- |
| `run_id` | STRING | Foreign reference to `ingestion_run_log.run_id` |
| `notebook_name` | STRING | Notebook that produced this error |
| `source_dataset_id` | STRING | Dataset being processed |
| `source_system` | STRING | e.g. `PSA` |
| `source_file` | STRING | Physical file name |
| `source_sheet` | STRING | Worksheet name (NULL if N/A) |
| `source_path` | STRING | Full source URL/path |
| `destination_table` | STRING | Intended Bronze table |
| `error_type` | STRING | Category of error |
| `error_message` | STRING | Exception or error message text |
| `error_timestamp` | TIMESTAMP | When the error occurred |
| `notes` | STRING | Free-text notes |

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ingestion_table('ingestion_errors')} (
    run_id             STRING,
    notebook_name      STRING,
    source_dataset_id  STRING,
    source_system      STRING,
    source_file        STRING,
    source_sheet       STRING,
    source_path        STRING,
    destination_table  STRING,
    error_type         STRING,
    error_message      STRING,
    error_timestamp    TIMESTAMP,
    notes              STRING
) USING DELTA
""")

# ── Validate ──
errors_df = spark.table(ingestion_table("ingestion_errors"))
print(f"✓ Table {ingestion_table('ingestion_errors')} created / verified.")
print(f"Rows: {errors_df.count()}")
errors_df.printSchema()
display(errors_df.limit(10))

✓ Table `tuklas_dev`.`01_ingestion`.`ingestion_errors` created / verified.
Rows: 0
root
 |-- run_id: string (nullable = true)
 |-- notebook_name: string (nullable = true)
 |-- source_dataset_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- source_path: string (nullable = true)
 |-- destination_table: string (nullable = true)
 |-- error_type: string (nullable = true)
 |-- error_message: string (nullable = true)
 |-- error_timestamp: timestamp (nullable = true)
 |-- notes: string (nullable = true)



run_id,notebook_name,source_dataset_id,source_system,source_file,source_sheet,source_path,destination_table,error_type,error_message,error_timestamp,notes


### Relationship Between the Three Tables

```
ingestion_run_log
    one row per notebook execution
           |
           | run_id
           v
file_manifest
    multiple rows per run
    one row per logical source/output
           |
           | failures
           v
ingestion_errors
    zero or more error records per run
```

```
01_ingest_psa
        |
        +--> ingestion_run_log
        |
        +--> file_manifest
        |
        +--> ingestion_errors
        |
        +--> actual data --> `02_bronze`.`psa_*`
```

Actual PSA/PSGC/DOT rows are **not** stored in the control tables — only metadata about ingestion runs, file processing results, and errors.

**Idempotency:** This notebook uses `CREATE TABLE IF NOT EXISTS` exclusively. It does not drop, truncate, delete, or overwrite any existing data. Safe to rerun.

In [0]:
from pyspark.sql.functions import lit

control_tables = [
    ("ingestion_run_log", "one row per ingestion notebook execution"),
    ("file_manifest",     "one row per logical source/output processed"),
    ("ingestion_errors",   "technical ingestion failures and exceptions"),
]

summary_rows = []
for tbl_name, purpose in control_tables:
    full_name = ingestion_table(tbl_name)
    exists = spark.catalog.tableExists(full_name)
    row_count = spark.table(full_name).count() if exists else -1
    summary_rows.append((tbl_name, purpose, exists, row_count))
    marker = "✓" if exists else "✗"
    print(f"{marker} {tbl_name:<22} | purpose: {purpose} | rows: {row_count}")

summary_df = spark.createDataFrame(
    summary_rows,
    schema="table_name STRING, purpose STRING, exists BOOLEAN, row_count BIGINT"
)
display(summary_df)

print("\nIngestion control setup complete.")
print(f"Schema: `{CATALOG}`.`{INGESTION_SCHEMA}`")
print("Tables: ingestion_run_log, file_manifest, ingestion_errors")

✓ ingestion_run_log      | purpose: one row per ingestion notebook execution | rows: 0
✓ file_manifest          | purpose: one row per logical source/output processed | rows: 0
✓ ingestion_errors       | purpose: technical ingestion failures and exceptions | rows: 0


table_name,purpose,exists,row_count
ingestion_run_log,one row per ingestion notebook execution,true,0
file_manifest,one row per logical source/output processed,true,0
ingestion_errors,technical ingestion failures and exceptions,true,0



Ingestion control setup complete.
Schema: `tuklas_dev`.`01_ingestion`
Tables: ingestion_run_log, file_manifest, ingestion_errors
